<a href="https://colab.research.google.com/github/BatoolAshour/PythonCodes/blob/main/ImageClassificationCNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

In [2]:
# Define transformations (convert to tensor)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

In [3]:
train_data= torchvision.datasets.CIFAR10(root='./data', train=True , transform= transform, download=True)
test_data= torchvision.datasets.CIFAR10(root='./data', train=False , transform= transform, download=True)

train_loader= torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
test_loader= torch.utils.data.DataLoader(test_data, batch_size=32, shuffle=True, num_workers=2)

100%|██████████| 170M/170M [00:13<00:00, 12.4MB/s]


In [4]:
image, label = train_data[0]

In [5]:
image.shape #3 Channels RGB, 32 * 32 Pixels

torch.Size([3, 32, 32])

In [6]:
class_names = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

In [7]:
class NeuralNet(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1= nn.Conv2d(in_channels=3, out_channels=12, kernel_size=5) # 32 - 5 = 27 / 1 = 27 + 1 = 28 --> New shape after convultional
    self.pool= nn.MaxPool2d(2, 2) # (12, 14, 14)
    self.conv2= nn.Conv2d(in_channels=12, out_channels=24, kernel_size=5) #14- 5 = 9+1 =10 --> New shape =  (24 ,10, 10) --> (24, 5, 5) --> Flatten (24*5*5)
    self.fc1= nn.Linear(24*5*5, 120)
    self.fc2= nn.Linear(120, 84)
    self.fc3 = nn.Linear(84, 10) #at the end you must have 10 output, the rest numbers is design choice

  def forward(self, x):
    x= self.pool(F.relu(self.conv1(x))) #Activation function break the Linearty
    x= self.pool(F.relu(self.conv2(x)))
    x= torch.flatten(x,1)
    x=F.relu(self.fc1(x))
    x=F.relu(self.fc2(x))
    x= self.fc3(x)
    return x


In [8]:
net = NeuralNet()
loss_function= nn.CrossEntropyLoss()
optimizer= optim.SGD(net.parameters(), lr=0.001, momentum=0.9) #Train The parameter

In [9]:

for epoch in range(30):
  print(f'Training epoch {epoch}...')

  running_loss= 0.0

  for i, data in enumerate(train_loader):
    inputs, labels = data

    optimizer.zero_grad()

    outputs = net(inputs)

    loss= loss_function(outputs, labels)
    loss.backward()
    optimizer.step()

    running_loss+= loss.item()


  print(f'Loss: {running_loss/ len(train_loader):.4f}')

Training epoch 0...
Loss: 2.2058
Training epoch 1...
Loss: 1.7952
Training epoch 2...
Loss: 1.5470
Training epoch 3...
Loss: 1.4134
Training epoch 4...
Loss: 1.3236
Training epoch 5...
Loss: 1.2441
Training epoch 6...
Loss: 1.1733
Training epoch 7...
Loss: 1.1094
Training epoch 8...
Loss: 1.0603
Training epoch 9...
Loss: 1.0140
Training epoch 10...
Loss: 0.9710
Training epoch 11...
Loss: 0.9359
Training epoch 12...
Loss: 0.8992
Training epoch 13...
Loss: 0.8720
Training epoch 14...
Loss: 0.8387
Training epoch 15...
Loss: 0.8106
Training epoch 16...
Loss: 0.7852
Training epoch 17...
Loss: 0.7581
Training epoch 18...
Loss: 0.7322
Training epoch 19...
Loss: 0.7077
Training epoch 20...
Loss: 0.6851
Training epoch 21...
Loss: 0.6622
Training epoch 22...
Loss: 0.6424
Training epoch 23...
Loss: 0.6160
Training epoch 24...
Loss: 0.5959
Training epoch 25...
Loss: 0.5785
Training epoch 26...
Loss: 0.5532
Training epoch 27...
Loss: 0.5355
Training epoch 28...
Loss: 0.5171
Training epoch 29...
Los

In [10]:
torch.save(net.state_dict(), 'trained_net.pth')

In [11]:
net= NeuralNet()
net.load_state_dict(torch.load('trained_net.pth'))

<All keys matched successfully>

In [14]:
correct= 0
total= 0

net.eval()

with torch.no_grad():
  for data in test_loader:
    images , labels = data
    outputs = net(images)
    _, predicted = torch.max(outputs, 1)
    total += labels.size(0)
    correct+= (predicted == labels).sum().item()



accuracy= 100* correct / total

print(f'Accuracy : {accuracy}%')

Accuracy : 68.83%


In [15]:
new_transform= transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

def load_image(image_path):
  image = Image.open(image_path)
  image = new_transform(image)
  image = image.unsqueeze(0)
  return image


image_paths= ['/content/example1.jpg', '/content/example2.jpg']

images = [load_image(img)for img in image_paths]

net.eval()

with torch.no_grad():
  for image in images:
    output= net(image)
    _, predicted = torch.max(output, 1)
    print(f'Prediction: {class_names[predicted.item()]}')

Prediction: dog
Prediction: plane
